In [1]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from datetime import date, datetime
import matplotlib.pyplot as plt
import mysql.connector as sq
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import mean_squared_error, f1_score
from sklearn.preprocessing import StandardScaler
import Levenshtein as lv
import re
from pymongo import MongoClient
from pymongo.errors import ServerSelectionTimeoutError

import seaborn as sns
import json
from sklearn.metrics import roc_auc_score, precision_score, recall_score
import os

# Cargamos las variables de entorno
load_dotenv()

# Lista de columnas
lst_col = [
    "dr0",
    "dr7",
    "dr15",
    "dr30",
    "dr60",
    "dr90",
    "dr180",
    "dr",
    #"yield"
]


lst_final_vars = [
'bank_name_shrinkage', # Lista
'tramo_amount_2_shrinkage', # Lista
'tramo_days_shrinkage', # Lista
'promo_code', # Lista (has_information_flag)
'ip_asn_flag_shrinkage', # Lista
'tramo_num_attempts_shrinkage', # Lista
'tramo_days_last_attempt_shrinkage', # Lista 
'req_ip_bin_shrinkage', # Lista
'last_attempt_flag_xgb_10_shrinkage',
'hour_loan_flag_shrinkage', # Lista
'day_week_flag_shrinkage', # Lista
'day_hour_loan_flag_shrinkage', # Lista
'device_browser_ver_flag_shrinkage',# Lista
'geo_consistency_score', # Lista

############################
###### TRUSTFULL VARS ######
############################
'digital_presence_score', # Lista TRUSTFULL
'breaches_count_email', # Lista (sin cambios) TRUSTFULL
'tramo_platforms_comercial_shrinkage', # Lista TRUSTFULL
'phone_has_twitter', # Lista TRUSTFULL
'email_has_image', # Lista TRUSTFULL
'whatsapp_privacy_status', # Lista (has_information_flag) TRUSTFULL
'same_name_phone_database', # Lista (match_names) TRUSTFULL
'phone_first_seen_days', # Lista (sin cambios) TRUSTFULL
'email_first_seen_days', # Lista (sin cambios) TRUSTFULL
'phone_has_telegram', # Lista TRUSTFULL
'tramo_platforms_network_tools_shrinkage', # Lista TRUSTFULL


############################
###### Similitud email #####
############################

# PREFIX feature
"email_min_lev_block_prefix", # Lista
"email_cnt_lev_le_1_block_prefix", # Lista
"email_block_size_prefix", # Lista
        
# SUFFIX features
"email_min_lev_block_suffix", # Lista
"email_cnt_lev_le_1_block_suffix", # Lista
"email_block_size_suffix", # Lista

############################
### Variables de tarjeta ###
############################

'transacciones_por_mes', # Lista
'tramo_n_categorias_distintas_shrinkage', # Lista
'n_chargebacks', # Lista
'tramo_fastloans_n_entidades_distintas_shrinkage', # Lista
'fl_min_diff_hours',# Lista
'amount_vs_fl_conc_7d',# Lista
'ratio_fl_concentration',# Lista
'bizzum_ratio',# Lista
'bizzum_intensity_velocity',# Lista
'mule_purity_check',# Lista
'bizzum_no_salary_risk',# Lista
]


lst_trust = [
#'digital_presence_score', # Lista TRUSTFULL
'phone_has_whatsapp', 'phone_has_instagram', 'phone_has_aliexpress',
'phone_has_telegram', 'phone_has_twitter', 'phone_has_weibo',
'email_has_spotify', 'email_has_linkedin', 'email_has_deliveroo',
'email_has_pinterest', 'email_has_wordpress', 'email_has_hubspot',
'email_has_gravatar', 'email_has_atlassian', 'email_has_lastpass',
'email_has_adobe', 'email_has_freelancer', 'email_has_github',
'email_has_disney_plus', 'email_has_google', 'email_has_duolingo',
'has_facebook', 'has_apple', 'has_amazon', 'has_office365', 'phone_first_name',


'breaches_count_email', # Lista (sin cambios) TRUSTFULL
#'tramo_platforms_comercial_shrinkage', # Lista TRUSTFULL

'email_has_image', # Lista TRUSTFULL
'whatsapp_privacy_status', # Lista (has_information_flag) TRUSTFULL
#'same_name_phone_database', # Lista (match_names) TRUSTFULL


'phone_first_seen_days', # Lista (sin cambios) TRUSTFULL
'email_first_seen_days', # Lista (sin cambios) TRUSTFULL
#'tramo_platforms_network_tools_shrinkage', # Lista TRUSTFULL
]

# Carga de nuevos usuarios 2026
## 1. Funciones
### 1.1 Funciones de carga de datos 

In [2]:
#########################################################################################################################################################
################################################################ FUNCIONES ##############################################################################
#########################################################################################################################################################
def get_connection():
    """
    Establece y devuelve una conexión a la base de datos MySQL utilizando las variables de entorno definidas.

    Returns
    -------
    mysql.connector.connection.MySQLConnection
        Objeto de conexión activo a la base de datos MySQL.
    """
    return sq.connect(
        host=os.getenv("DB_HOST"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database=os.getenv("DB_NAME"),
        use_pure=True,   
    )

def get_querys(lst_query):
    """
    Función encargada de realizar las querys y devolver los datos de estas consultas en formato de DataFrame.
    
    Parameters
    ----------
    lst_query (list): Lista con todas las querys que vamos a realizar.

    Returns
    -------
    pandas.DataFrame: DataFrame con los resultados de las consultas.
    """
    print("Comenzamos a hacer las consultas")
    # Lista con todos los df
    lst_df = []
    # Establecemos la conexion
    conn = get_connection()
    # Creamos el cursor para ejecutar las queries
    cur = conn.cursor()

    for query in lst_query:
        cur.execute(query)
        # Guardamos el resultado
        table_result = cur.fetchall()
        column_names = [desc[0] for desc in cur.description] # Guardamos los nombres de las columnas
        # Creamos un df a partir del resultado
        lst_df.append(pd.DataFrame(table_result, columns=column_names))

    cur.close()
    conn.close()

    print("✅ Consultas finalizadas")
    return lst_df

    
def shrinkage(df, var, dr='dr90'):

    # Calculamos la cantidad de datos de cada uno de las categorias de var
    df_count = df[var].value_counts().reset_index()

    # Media global de DR
    global_dr = df[dr].mean()

    # Calculamos la media de cada uno de los subgrupos
    df_mean = df.groupby(var)[dr].mean().reset_index()

    # Combinamos los dos df donde tenemos la media de cada uno de los bancos
    df_mean = df_mean.merge(df_count, on=var, how='left') 

    df_mean[var+"_shrinkage"] = ( (df_mean["count"] * df_mean[dr]) + (1000*global_dr)) / (global_dr + df_mean["count"])

    if var+"_shrinkage" in df.columns:
        df.drop(var+"_shrinkage", axis=1, inplace=True)

    df = df.merge(df_mean[[var, var+"_shrinkage"]], on=var, how='left')

    return df

def var_mean(df, var, dr='dr90'):

    # Calculamos la cantidad de datos de cada uno de las categorias de var
    df_count = df[var].value_counts().reset_index()

    # Calculamos la media de cada uno de los subgrupos
    df_mean = df.groupby(var)[dr].mean().reset_index().rename(columns={dr: var+"_shrinkage"})

    # Combinamos los dos df donde tenemos la media de cada uno de los bancos
    df_mean = df_mean.merge(df_count, on=var, how='left') 

    df = df.merge(df_mean[[var,  var+"_shrinkage"]], on=var, how='left')

    return df

    
def safe_bool(v):
    # Normaliza None / bool / 0-1 / "true"/"false"
    if v is None:
        return 0

    if isinstance(v, bool):
        return int(v)

    if isinstance(v, (int, float)):
        return int(v != 0)

    if isinstance(v, str):
        return int(v.strip().lower() in ("1", "true", "yes", "y", "t"))

    return int(bool(v))


def calculate_num_platforms(df, lst_cols_trust, var_name):

    df[var_name] = (
        df[lst_cols_trust]
        .fillna(0)
        .astype("int8")
        .sum(axis=1)
    )

    return df


def calculate_tramo_creation(df, var, bins, labels, tramo_var='tramo'):
    df[tramo_var] = pd.cut(
        df[var],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    return df

def prep_catboost(X, cat_cols):
    X = X.copy()
    for c in cat_cols:
        # Rellena NaN y convierte a string
        X[c] = X[c].astype("object").fillna("__MISSING__").astype(str)
    return X

def match_names( name1, name2):
    if name1 in name2:
        return 1
    elif name2 in name1:
        return 1
    return 0 

def auc_gini(df, var, dr, threshold=0.8):
    df = df.copy()
    df[dr] = np.where(df[dr]>=threshold,1,0)
    y = df[dr].astype(int).values
    X = df[var].astype(str)

    p_global = y.mean()

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    oof_score = np.zeros(len(df), dtype=float)

    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr = y[tr_idx]

        # media por banco SOLO en train
        bank_rate = pd.Series(y_tr, index=X_tr).groupby(level=0).mean()

        # score para valid: si no existe el banco en train -> global
        oof_score[va_idx] = X_va.map(bank_rate).fillna(p_global).values

    auc = roc_auc_score(y, oof_score)
    gini = 2*auc - 1
    return auc, gini

def full_analysis_by_var(df, var, dr, title, dr_evolution=False, calculate_gini=False):
    """
    Ejecuta el análisis completo de una variable:
    - Lift plot con CI
    - Evolución temporal del DR
    - AUC y Gini
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de entrada
    var : str
        Variable categórica / binning (ej. 'tramo_platforms')
    dr : str
        Variable de default binaria (ej. 'dr90')
    title : str
        Título del gráfico temporal
    
    Returns
    -------
    results : dict
        Diccionario con AUC y Gini
    """
    print("#" * 150)
    print("#" * 150)
    print(f"\n📊 Análisis de variable: {var}")
    print("-" * 50)

    # 1️⃣ Lift + CI
    df_lift = plot_lift_by_var(df, dr, var)

    # 2️⃣ Evolución temporal
    line_plot(
        df,
        var,
        lst_col,
        title
    )
    if calculate_gini:
        auc, gini = auc_gini(df, var, dr)

        print(f"\n📈 Métricas discriminación")
        print(f"AUC  : {auc:.4f}")
        print(f"Gini : {gini:.4f}")

    # Evolucion mes a mes del dr segun si tiene o no 
    if dr_evolution :
        graph_dr_by_bank(df, df[var].unique().tolist(), lst_col, var)


    return {
        "variable": var,
        "dr": dr,
        "lift_table": df_lift
    }


def calculate_ratios(df):

    real_profit = round(df["profit"].sum(),2)
    real_vol = df.shape[0]

    # Inicializamos un df vacío con las columnas correspondientes
    df_table = pd.DataFrame(
        data = [[""] *7], 
        columns=
        [
            "Model", 
            "Vol_con_clusters", 
            "Vol_sin_clusters", 
            "Profit_con_clusters (€)",
            "Profit_sin_clusters (€)",
            "Ratio_vol", 
            "Diff_profit (€)"
        ]
    )
    
    # Filtramos y eliminamos aquellos usuarios marcados como fraude 
    df_cluster = df[df["fraud_prediction"]==0]

    cluster_profit = round(df_cluster["profit"].sum(), 2)
    cluster_vol = df_cluster.shape[0]
    ratio_vol = round((cluster_vol / real_vol), 2)
    diff_profit = round((cluster_profit - real_profit), 2)

    # Añadimos los resultados a las tablas
    df_table.loc[df_table.shape[0]] = ["Modelo actual", real_vol, cluster_vol, real_profit, cluster_profit, ratio_vol, diff_profit]
        
        # print("#######################################################")
        # print(f"Representa el {ratio_vol * 100} de la muestra de nuevos clientes")
        # line_plot_no_fraud(df_cluster, f"Evolucion buenos usuarios sin {combi["email"]} y {combi["phone"]}")
    return df_table.iloc[1:]


def normalize_email(email):
    if not isinstance(email, str) or '@' not in email:
        return ""
    email = email.strip().lower()
    local, domain = email.split('@', 1)

    # quitar +tag
    local = local.split('+', 1)[0]

    # regla de puntos solo para gmail/googlemail
    if domain in ("gmail.com", "googlemail.com"):
        local = local.replace('.', '')

    # colapsar separadores repetidos
    local = re.sub(r'[\._-]+', '.', local)

    return f"{local}@{domain}"
def get_db(uri, db):
    client = MongoClient(
        uri,
        serverSelectionTimeoutMS=5000,
        appname="qb-prod-trustfull",
    )
    # ping para validar conexión
    client.admin.command("ping")
    return client, client[db]

### 1.2 Funciona de graficos

In [3]:

#########################################################################################################################################################
################################################################ GRÁFICOS ###############################################################################
#########################################################################################################################################################
def plot_lift_by_var(df, dr, var):
    """
    Calcula lift y dibuja DR por tramos con intervalo de confianza.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de entrada
    dr : str
        Nombre de la columna de default rate (binaria 0/1)
    var : str
        Variable categórica por la que agrupar
    
    Returns
    -------
    df_lift : pd.DataFrame
        DataFrame con n_users, dr, lift, se, ci_low, ci_high
    """

    # DR global
    p_global = df[dr].mean()

    # Agregación
    df_lift = (
        df.groupby(var)
        .agg(
            n_users=(dr, 'count'),
            dr=(dr, 'mean')
        )
        .reset_index()
    )

    # Orden (por categoría)
    df_lift = df_lift.sort_values(by=var, ascending=False)

    # Lift
    df_lift["lift"] = df_lift["dr"] / p_global

    # Error estándar y CI
    df_lift["se"] = np.sqrt(
        df_lift["dr"] * (1 - df_lift["dr"]) / df_lift["n_users"]
    )

    df_lift["ci_low"] = df_lift["dr"] - 1.96 * df_lift["se"]
    df_lift["ci_high"] = df_lift["dr"] + 1.96 * df_lift["se"]
    df_lift["ci_width"] = np.where(
        (df_lift["ci_high"] - df_lift["ci_low"]) < 0.03,
        "very_stable",
        np.where(
            (df_lift["ci_high"] - df_lift["ci_low"]) < 0.06,
            "acceptable",
            "very_unestable"
        )
    )

    # Plot
    plt.figure(figsize=(8, 5))
    plt.errorbar(
        df_lift["dr"],
        df_lift[var],
        xerr=1.96 * df_lift["se"],
        fmt="o"
    )
    
    plt.axvline(p_global, color='red', linestyle='--', label='Global DR')
    plt.legend()
    plt.xlabel('Default rate')
    plt.show()

    return df_lift

    
def line_plot(df, col_cluster, lst_col, title="Evolución del Default Rate por Cluster"):
    """
    Genera un gráfico de líneas donde cada línea representa
    la evolución de DRs para un cluster distinto.
    """

    # Agrupación por cluster
    df_group = df.groupby(col_cluster)[lst_col].mean()

    # Convertimos a formato largo (melt)
    df_plot = df_group.reset_index().melt(
        id_vars=col_cluster,
        value_vars=lst_col,
        var_name="dr_day",
        value_name="value"
    )

    # Plot
    plt.figure(figsize=(12, 6))
    for cluster, data in df_plot.groupby(col_cluster):
        plt.plot(
            data["dr_day"], 
            data["value"], 
            marker="o", 
            label=str(cluster)
        )

        # Añadir etiquetas
        for x, y in zip(data["dr_day"], data["value"]):
            plt.text(
                x, y, f"{y:.2f}",
                ha="center", va="bottom", fontsize=8
            )

    plt.title(title)
    plt.xlabel("DR Day")
    plt.ylabel("Valor DR")
    plt.grid(True, alpha=0.3)
    plt.legend(title=col_cluster)
    plt.tight_layout()
    plt.show()


def graph_dr_by_bank(df, var_list, dr_cols, var):

    df = df.copy()
    df["year_month_of_creation"] = pd.to_datetime(df["year_month_of_creation"])

    for bank in var_list:
        df_bank = df[df[var] == bank]
        percent = (df_bank.shape[0] / df.shape[0]) * 100
        print(f"{bank} representa el {percent:.2f}% de la muestra")
        

        plt.figure(figsize=(12, 5))
        plt.title(f"Evolución DR segun si tiene Whatssap – {bank}")
        plt.xlabel("Mes")
        plt.ylabel("Default Rate")

        for col in dr_cols:
            # Calculamos para cada columna drx cual es la media por banco 
            df_bank_group = df_bank.groupby("year_month_of_creation")[col].mean().reset_index()
            # Color especial para profit
            if col == "profit":
                color = "#006400"  # Verde oscuro fuerte
                zorder = 5         # Para que destaque por encima
                linewidth = 2.5    # Más gruesa
            else:
                color = None       # Color automático
                zorder = 2
                linewidth = 1.5

            # Dibujar línea
            plt.plot(
                df_bank_group["year_month_of_creation"],
                df_bank_group[col],
                marker="o",
                label=col,
                alpha=0.85,
                color=color,
                linewidth=linewidth,
                zorder=zorder
            )

            # Etiquetas sobre los puntos
            for x, y in zip(df_bank_group["year_month_of_creation"], df_bank_group[col]):
                plt.text(
                    x, y, f"{y:.2f}" if col in("dr", "dr0", "profit")else "",
                    fontsize=8,
                    ha="center",
                    va="bottom"
                )

        plt.legend()
        plt.tight_layout()
        plt.show()


def line_plot_model_impact(
    df,
    fraud_flag_col,          # columna booleana / 0-1: 1 si el modelo marca fraude
    lst_col,                 # columnas dr_day: ["dr_7", "dr_14", ...]
    title="Impacto del modelo en la evolución del Default Rate",
    label_off="Modelo OFF (todos)",
    label_on="Modelo ON (excluye fraude)"
):
    """
    Compara la evolución de DR entre:
      - Modelo OFF: toda la población
      - Modelo ON: excluyendo usuarios marcados como fraude (fraud_flag_col==1)

    Dibuja 2 líneas (OFF vs ON) con etiquetas en cada punto.
    """

    # Escenario OFF: todos
    df_off = df.copy()

    # Escenario ON: quitamos fraude (marcados por el modelo)
    df_on = df[df[fraud_flag_col] == 0].copy()

    # Medias de DR por escenario (sin groupby adicional)
    s_off = df_off[lst_col].mean()
    s_on  = df_on[lst_col].mean()

    # Plot
    plt.figure(figsize=(12, 6))

    x = list(s_off.index)
    y_off = list(s_off.values)
    y_on  = list(s_on.values)

    plt.plot(x, y_off, marker="o", label=label_off)
    plt.plot(x, y_on, marker="o", label=label_on)

    # Etiquetas
    for xi, yi in zip(x, y_off):
        plt.text(xi, yi, f"{yi:.2f}", ha="center", va="bottom", fontsize=8)
    for xi, yi in zip(x, y_on):
        plt.text(xi, yi, f"{yi:.2f}", ha="center", va="top", fontsize=8)

    plt.title(title)
    plt.xlabel("DR Day")
    plt.ylabel("Default Rate")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 2. Carga de Datos

In [ ]:
query = """
select  
    date_format(loans.created_at, '%Y-%m') year_month_of_creation,
    la.created_at,
    COALESCE(banks.name, db.issuer) as bank_name,
    banks.bank_code, la.bank_account_id as id_bank, 
    u.public_id,
    la.ip_address, la.id, la.user_id, la.user_agent as device_info, 
    la.promo_code_id, la.days,
    loans.ref_id, loans.status, loans.amount, loans.loan_number,
    cd.cell_phone, cd.email, u.dni, ad.city,
    pd.first_name, pd.last_name,pd.educational_level, pd.home_status, pd.marital_status, pd.nationality, pd.owns_driving_license, pd.gender,
    if(p.collections_0 is null, 0, p.collections_0) collections_0, 
    if(p.collections_7 is null, 0, p.collections_7) collections_7, 
    if(p.collections_15 is null, 0, p.collections_15) collections_15,
    if(p.collections_30 is null, 0, p.collections_30) collections_30,
    if(p.collections_60 is null, 0, p.collections_60) collections_60,
    if(p.collections_90 is null, 0, p.collections_90) collections_90,
    if(p.collections_180 is null, 0, p.collections_180) collections_180,
    if(p.amortized_capital is null, 0, p.amortized_capital) principal_collected,
    if(p.total_collected is null, 0, p.total_collected) total_collected,
    CASE 
    WHEN la.transfer_date IS NULL OR la.created_at IS NULL THEN NULL
        ELSE TIMESTAMPDIFF(MINUTE,la.created_at, la.transfer_date)+1
    END AS diff_minutes

from loan_applications la
left join loans on la.id=loans.loan_application_id
left join contact_details cd on cd.id = la.contact_details_id
left join personal_details pd on pd.id = la.personal_details_id
left join users u on u.personal_details_id = pd.id
left join debit_cards db on db.id = la.debit_card_id
left join addresses ad on ad.id = la.address_id
left join (
    select 
        p.loan_application_id, 
        sum(p.amort_capital) amortized_capital, 
        sum(p.amount) total_collected, 
        sum(if(datediff(p.paid_at, la.due_date)<=0, amort_capital, 0)) collections_0, 
        sum(if(datediff(p.paid_at, la.due_date)<=7, amort_capital, 0)) collections_7, 
        sum(if(datediff(p.paid_at, la.due_date)<=15, amort_capital, 0)) collections_15,
        sum(if(datediff(p.paid_at, la.due_date)<=30, amort_capital, 0)) collections_30,
        sum(if(datediff(p.paid_at, la.due_date)<=60, amort_capital, 0)) collections_60,
        sum(if(datediff(p.paid_at, la.due_date)<=90, amort_capital, 0)) collections_90,
        sum(if(datediff(p.paid_at, la.due_date)<=180, amort_capital, 0)) collections_180
    from payments p
    left join loan_applications la on p.loan_application_id=la.id
    where p.status='success'
    group by 1
) p on p.loan_application_id=la.id

left  JOIN(
    SELECT ba.id,b.name,b.commercial_name, ba.bank_code
    FROM bank_accounts ba 
    LEFT JOIN(
        SELECT banks.*,banks.code COLLATE utf8_unicode_ci AS bcode
        FROM banks
    ) b ON ba.bank_code = b.bcode
) banks ON banks.id = la.bank_account_id



where loans.status in ('closed', 'delinquent', 'asnef', 'fraud', 'write_off')
and la.created_at >= '2026-01-14 00:00:00'
and la.created_at <= '2026-03-31 23:59:59'
AND is_returning = false 
"""

# df_26 = get_querys([query])[0]
df_26 = pd.read_csv("../data/datos/df_26_raw.csv")
# df_26.to_csv("../data/datos/df_26_raw.csv", index=False)

In [5]:
df_26["amount"] = pd.to_numeric(df_26["amount"], errors="coerce")
df_26["loan_number"] = pd.to_numeric(df_26["loan_number"], errors="coerce")
df_26["collections_0"] = pd.to_numeric(df_26["collections_0"], errors="coerce")
df_26["collections_7"] = pd.to_numeric(df_26["collections_7"], errors="coerce")
df_26["collections_15"] = pd.to_numeric(df_26["collections_15"], errors="coerce")
df_26["collections_30"] = pd.to_numeric(df_26["collections_30"], errors="coerce")
df_26["collections_60"] = pd.to_numeric(df_26["collections_60"], errors="coerce")
df_26["collections_90"] = pd.to_numeric(df_26["collections_90"], errors="coerce")
df_26["collections_180"] = pd.to_numeric(df_26["collections_180"], errors="coerce")
df_26["principal_collected"] = pd.to_numeric(df_26["principal_collected"], errors="coerce")
df_26["total_collected"] = pd.to_numeric(df_26["total_collected"], errors="coerce")

# calculamos los DR diferentes, el profit y el yield
df_26["dr0"] = (df_26["amount"] - df_26["collections_0"]) / df_26["amount"]
df_26["dr7"] = (df_26["amount"] - df_26["collections_7"]) / df_26["amount"]
df_26["dr15"] = (df_26["amount"] - df_26["collections_15"]) / df_26["amount"]
df_26["dr30"] = (df_26["amount"] - df_26["collections_30"]) / df_26["amount"] 
df_26["dr60"] = (df_26["amount"] - df_26["collections_60"]) / df_26["amount"]
df_26["dr90"] = (df_26["amount"] - df_26["collections_90"]) / df_26["amount"]
df_26["dr180"] = (df_26["amount"] - df_26["collections_180"]) / df_26["amount"]
df_26["dr"] = (df_26["amount"] - df_26["principal_collected"]) / df_26["amount"]
df_26["profit"] = (df_26["total_collected"] - df_26["amount"])
df_26["yield"] = (df_26["total_collected"] - df_26["amount"]) / df_26["amount"]

In [11]:
# Obtencion de las variables de trustfull 
MONGO_URI= os.getenv("MONGO_URI_PROD_DEST")  
MONGO_DB_TRUST= os.getenv("MONGO_DB_TRUST") 

client, db = get_db(MONGO_URI, MONGO_DB_TRUST)

# 1. Obtener la lista de IDs públicos únicos del DF
public_ids = df_26['public_id'].dropna().unique().tolist()

# 2. Acceder a la colección específica
# Nota: Según tu descripción, la colección es 'contact_details_confidence'
collection = db['contact_details_confidence']

def extract_mongo_vars(public_id, collection, lst_trust):
    """
    Busca un customer_id en Mongo y extrae dinámicamente las variables de trustfull.
    """
    # Inicializamos con None para todas las columnas de la lista
    default_resp = [None] * len(lst_trust)
    
    if pd.isna(public_id):
        return pd.Series(default_resp)

    doc = collection.find_one({"customer_id": public_id})
    if not doc:
        return pd.Series(default_resp)

    # Extraemos los bloques principales una sola vez
    claims = doc.get('claims', {})
    # Asumimos que dentro de email/phone hay una lista y tomamos el primer elemento [0]
    # Si no hay lista, el .get([], [{}])[0] evita errores de índice
    e_data = claims.get('email', [{}])[0] if isinstance(claims.get('email'), list) else {}
    p_data = claims.get('phone', [{}])[0] if isinstance(claims.get('phone'), list) else {}

    results = []
    
    for var in lst_trust:
        try:
            # CASO 1: Variables compartidas (OR logic)
            if var in ['has_facebook', 'has_apple', 'has_amazon', 'has_office365']:
                val_e = e_data.get(var)
                val_p = p_data.get(var)
                # Si alguno es True, el resultado es True. Si ambos son None, queda None.
                if val_e is True or val_p is True:
                    val = True
                elif val_e is False or val_p is False:
                    val = False
                else:
                    val = None
            
            # CASO 2: Específicas de Phone (incluyendo whatsapp_privacy_status)
            elif var.startswith('phone_') or var == 'whatsapp_privacy_status':
                # Si empieza por phone_, quitamos el prefijo para buscar en la clave de Mongo
                # Ejemplo: phone_has_whatsapp -> tiene que buscar 'has_whatsapp'? 
                # Si en Mongo la clave es exactamente 'phone_has_whatsapp', no quites el replace.
                key = var.replace('phone_', '') if var.startswith('phone_') else var
                val = p_data.get(key)

            # CASO 3: Específicas de Email
            elif var.startswith('email_') or var == 'breaches_count_email':
                key = var.replace('email_', '') if var.startswith('email_') else var
                val = e_data.get(key)
            
            # CASO 4: Otros
            else:
                val = doc.get(var) # Búsqueda en raíz por si acaso

            results.append(val)
            
        except Exception:
            results.append(None)

    return pd.Series(results)

df_26[lst_trust] = df_26['public_id'].apply(
    lambda x: extract_mongo_vars(x, collection, lst_trust)
)

In [21]:
df_26[['public_id']+lst_trust]


,public_id,phone_has_whatsapp,phone_has_instagram,phone_has_aliexpress,phone_has_telegram,phone_has_twitter,phone_has_weibo,email_has_spotify,email_has_linkedin,email_has_deliveroo,...,has_facebook,has_apple,has_amazon,has_office365,phone_first_name,breaches_count_email,email_has_image,whatsapp_privacy_status,phone_first_seen_days,email_first_seen_days
0,ce2c488c-2a10-4a10-8cef-50ae24b413cb,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,e4281434-d7ba-44b4-9004-46ebd13f5522,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,f8d79b87-11d8-4c5a-9868-dfcb2c67cb65,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,627a1fe3-8871-4c6e-8e65-c8679bc315ec,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,deb14db8-fa99-4b10-a665-2cba9ef2ddac,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6330,9228a80e-9220-4f71-8832-cdafa738a0b6,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
6331,6a163115-9aa7-42f3-b0a8-cfb07c1cedda,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
6332,cba968b1-4696-4b8f-aa6b-fb018dae7577,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
6333,2fe341c8-eb71-4f24-8680-191f738af2a5,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [10]:
# Obtiene el primer documento que encuentre en la colección
documento_ejemplo = collection.find_one()

# Para verlo bien estructurado (con sangrías)
import json
from bson import json_util

print(json.dumps(documento_ejemplo, indent=4, default=json_util.default))

{
    "_id": {
        "$oid": "69661b63017aee1a05aa1d59"
    },
    "resolution_id": "14851a16-0abd-4ecc-9f76-995d673fc88f",
    "customer_id": "eb94a431-1dea-491d-9e1f-6f25cbccf4fd",
    "device_request_time": "2026-01-13T10:15:56.982Z",
    "provider": "Trustfull",
    "claims": {
        "email": {
            "value": "sakinehamadi22@gmail.com",
            "is_disposable": false,
            "score": 743,
            "score_cluster": "high",
            "resolution_id": "14851a16-0abd-4ecc-9f76-995d673fc88f",
            "device_request_time": "2026-01-13T10:15:56.982Z",
            "data_breaches_count": 1,
            "data_breaches_first_breach": "2022-10-20",
            "data_breaches_last_breach": "2022-10-20",
            "data_breaches_list": "ClickASnap,2022-10-20",
            "data_breaches_data": "[{\"id\":\"39773\",\"date\":\"2022-10-20\",\"title\":\"ClickASnap\"}]",
            "data_breaches_data_lists": "{\"names\":[],\"emails\":[],\"phones\":[]}",
            "ha

In [16]:
df_26['public_id']

0       ce2c488c-2a10-4a10-8cef-50ae24b413cb
1       e4281434-d7ba-44b4-9004-46ebd13f5522
2       f8d79b87-11d8-4c5a-9868-dfcb2c67cb65
3       627a1fe3-8871-4c6e-8e65-c8679bc315ec
4       deb14db8-fa99-4b10-a665-2cba9ef2ddac
                        ...                 
6330    9228a80e-9220-4f71-8832-cdafa738a0b6
6331    6a163115-9aa7-42f3-b0a8-cfb07c1cedda
6332    cba968b1-4696-4b8f-aa6b-fb018dae7577
6333    2fe341c8-eb71-4f24-8680-191f738af2a5
6334    85cef7e9-6e87-4a64-b84c-52cda6cda696
Name: public_id, Length: 6335, dtype: object

In [20]:
test_id = "9228a80e-9220-4f71-8832-cdafa738a0b6"
print(collection.find_one({"customer_id": test_id}))

{'_id': ObjectId('69cac635b38bf44e7f194e3d'), 'resolution_id': '1a7d07f2-746b-46a3-a6e6-b1df82ed7d1e', 'customer_id': '9228a80e-9220-4f71-8832-cdafa738a0b6', 'device_request_time': '2026-03-30T18:51:26.486Z', 'provider': 'Trustfull', 'claims': {'email': {'value': 'juancarlossaezgil99@gmail.com', 'is_disposable': False, 'score': 730, 'score_cluster': 'high', 'resolution_id': '1a7d07f2-746b-46a3-a6e6-b1df82ed7d1e', 'device_request_time': '2026-03-30T18:51:26.486Z', 'data_breaches_count': 6, 'data_breaches_first_breach': '2017-08-24', 'data_breaches_last_breach': '2023-02-09', 'data_breaches_list': 'JoyGames Combolist,2023-02-09|Sensitive Source,2021-09-09|Sensitive Source,2021-03-25|January 2021 Active Combo List,2021-01-28|Soar Games,2020-08-13|Taringa,2017-08-24', 'data_breaches_data': '[{"id":"41326","date":"2023-02-09","title":"JoyGames Combolist"},{"id":"37922","date":"2021-09-09","title":"Sensitive Source"},{"id":"37314","date":"2021-03-25","title":"Sensitive Source"},{"id":"37173"

In [22]:
df_26['created_at']

0       2026-01-13 00:04:14.159267
1       2026-01-13 00:44:17.811127
2       2026-01-13 00:46:14.777809
3       2026-01-13 00:52:12.471181
4       2026-01-13 01:10:14.038655
                   ...            
6330    2026-03-30 20:58:12.213193
6331    2026-03-30 21:26:13.941762
6332    2026-03-30 21:46:15.542538
6333    2026-03-30 23:18:13.613172
6334    2026-03-30 23:34:30.392461
Name: created_at, Length: 6335, dtype: object